# LangChain Hands-on 05 — Memory & Chat History
**Trainer:** Ajeetkumar

**What we will practise**
- LLMs are stateless (they forget)
- Manual memory with a list
- InMemoryChatMessageHistory
- RunnableWithMessageHistory (memory per user / session)
- Save chat to a file
- Trimming history to save tokens
- Chatbot loop

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-community

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

## 1. Problem — the LLM forgets

In [ ]:
print(llm.invoke("Hi, my name is Ajeet and I live in Pune.").content)

In [ ]:
print(llm.invoke("What is my name and where do I live?").content)

## 2. Manual memory — keep a list, send it every time

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
chat_history = [SystemMessage("You are a friendly assistant. Keep answers short.")]

def chat(user_input):
    chat_history.append(HumanMessage(user_input))
    response = llm.invoke(chat_history)
    chat_history.append(AIMessage(response.content))
    return response.content

In [ ]:
chat("Hi, my name is Ajeet and I live in Pune.")

In [ ]:
chat("What is my name and where do I live?")

In [ ]:
len(chat_history), chat_history

## 3. InMemoryChatMessageHistory — a ready-made history object

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

In [ ]:
history = InMemoryChatMessageHistory()
history.add_user_message("I love cricket")
history.add_ai_message("Great! Who is your favourite player?")
history.messages

## 4. RunnableWithMessageHistory — automatic memory per session

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Keep replies under 60 words."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm

In [ ]:
store = {}     # session_id -> history

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
chatbot = RunnableWithMessageHistory(
    chain, get_session_history,
    input_messages_key="input", history_messages_key="history",
)

In [ ]:
config = {"configurable": {"session_id": "ajeet"}}
chatbot.invoke({"input": "I want a 3-day trip from Pune with Rs 15,000 budget."}, config=config).content

In [ ]:
chatbot.invoke({"input": "I like mountains."}, config=config).content

In [ ]:
chatbot.invoke({"input": "Summarise my requirements."}, config=config).content

### Different user = different memory

In [ ]:
chatbot.invoke({"input": "Summarise my requirements."}, config={"configurable": {"session_id": "priya"}}).content

In [ ]:
{k: len(v.messages) for k, v in store.items()}

## 5. Save chat history in a file (survives restart)

In [ ]:
from langchain_community.chat_message_histories import FileChatMessageHistory

In [ ]:
file_bot = RunnableWithMessageHistory(
    chain, lambda session_id: FileChatMessageHistory(f"chat_{session_id}.json"),
    input_messages_key="input", history_messages_key="history",
)
file_bot.invoke({"input": "Remember: my favourite city is Shimla."}, config={"configurable": {"session_id": "u1"}}).content

In [ ]:
print(open("chat_u1.json").read()[:500])

## 6. Trim history — send only recent messages (save tokens & cost)

In [ ]:
from langchain.messages import trim_messages

In [ ]:
long_history = store["ajeet"].messages
trimmed = trim_messages(long_history, max_tokens=80, strategy="last",
                        token_counter="approximate", start_on="human")
print(len(long_history), "→", len(trimmed))
for m in trimmed:
    print(m.type, ":", m.content[:60])

# Memory types (theory)
| Type | Idea | Good for |
|---|---|---|
| Buffer | keep all messages | short chats |
| Window | keep last k messages | long chats, low cost |
| Summary | LLM summarises old messages | very long chats |
| Vector / long-term | save facts in a DB | personal assistants |

> In LangChain v1 the recommended memory for **agents** is a **checkpointer + thread_id** (notebook 13). Old `ConversationBufferMemory` is in `langchain-classic` (notebook 16).

## 7. Simple chatbot loop (type `exit` to stop)

In [ ]:
config = {"configurable": {"session_id": "live_demo"}}
while True:
    user = input("You: ")
    if user.lower() in ["exit", "quit"]:
        break
    print("Bot:", chatbot.invoke({"input": user}, config=config).content)

## 🧪 Practice
1. Build a "Doctor appointment" bot that remembers name, age and symptoms.
2. Keep only the last 4 messages using `trim_messages` inside the chain.
3. Show that two session ids keep separate memories.